# 1. Exploration et ingestion

In [16]:
import os
import locale
import json

import kagglehub
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

locale.setlocale(locale.LC_ALL, 'fr_FR.UTF-8')

'fr_FR.UTF-8'

In [2]:
from dotenv import load_dotenv

# Charge les variables (identifiants Kaggle) depuis le fichier .env
load_dotenv()

True

In [17]:
OUT_DIR = 'data'

os.makedirs(OUT_DIR, exist_ok=True)

## 1.1 Chargement des données

https://www.kaggle.com/datasets/sdolezel/black-friday

In [3]:
file_name = "train.csv"

In [4]:
# téléchargement du dataset
path = kagglehub.dataset_download("sdolezel/black-friday")

print("Path to dataset files:", path)

Path to dataset files: C:\Users\p02114\.cache\kagglehub\datasets\sdolezel\black-friday\versions\1


In [5]:
dataset_path = Path(path) / file_name

if dataset_path.exists():
    df = pd.read_csv(dataset_path)
    print("Dataset chargé avec succès.")
else:
    raise FileNotFoundError(f"""Le chemin spécifié est invalide: '{dataset_path.as_posix()}'.
        Vérifiez si le fichier a été téléchargé avec succès.""")

Dataset chargé avec succès.


In [6]:
# Dimensions du dataset
print(f"Nombre de lignes : {df.shape[0]:n}")
print(f"Nombre de colonnes : {df.shape[1]:n}")

Nombre de lignes : 550 068
Nombre de colonnes : 12


In [7]:
df.head()

,User_ID,Product_ID,Gender,Age,Occupation,City_Category,Stay_In_Current_City_Years,Marital_Status,Product_Category_1,Product_Category_2,Product_Category_3,Purchase
0,1000001,P00069042,F,0-17,10,A,2,0,3,NaN,NaN,8370
1,1000001,P00248942,F,0-17,10,A,2,0,1,6.0,14.0,15200
2,1000001,P00087842,F,0-17,10,A,2,0,12,NaN,NaN,1422
3,1000001,P00085442,F,0-17,10,A,2,0,12,14.0,NaN,1057
4,1000002,P00285442,M,55+,16,C,4+,0,8,NaN,NaN,7969


## 1.2 EDA

__Features :__
- `User_ID` : ID utilisateur
- `Product_ID` : ID produit
- `Gender` : sexe de l'utilisateur
- `Age` : tranches d'âges
- `Occupation` : occupation de l'utilisateur (anonymisé)
- `City_Category` : catégorie de la ville (A, B, C - ordinal ?)
- `Stay_In_Current_City_Years` : nombre d'années passées dans la ville actuelle
- `Marital_Status` : statut marital (marié 1 ou non 0)
- `Product_Category_1` : catégorie principale du produit (anonymisé)
- `Product_Category_2` : 2ème catégorie du produit (anonymisé)
- `Product_Category_3` : 3ème catégorie du produit (anonymisé)
- `Purchase` : quantité achetée

*NB : un produit peut avoir plusieurs catégories*

In [8]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 550068 entries, 0 to 550067
Data columns (total 12 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   User_ID                     550068 non-null  int64  
 1   Product_ID                  550068 non-null  str    
 2   Gender                      550068 non-null  str    
 3   Age                         550068 non-null  str    
 4   Occupation                  550068 non-null  int64  
 5   City_Category               550068 non-null  str    
 6   Stay_In_Current_City_Years  550068 non-null  str    
 7   Marital_Status              550068 non-null  int64  
 8   Product_Category_1          550068 non-null  int64  
 9   Product_Category_2          376430 non-null  float64
 10  Product_Category_3          166821 non-null  float64
 11  Purchase                    550068 non-null  int64  
dtypes: float64(2), int64(5), str(5)
memory usage: 50.4 MB


En réalité, hormis `Purchase` la plupart des variables numériques sont en réalité des varaibles catégorielles encodées (`Occupation`, `Marital_Status`, `Product_Category_X`). De plus, les variables `Product_Category_2` et `Product_Category_3` sont sans doute des `float`à cause des valeurs manquantes.

Enfin, nous avons des types `str` pour les variables que nous aurions, à priori, classés en tant que numériques `Age` et `Stay_In_Current_City_Years` car :
1. `Age` est un regroupement par tranches d'âges
2. les gens qui sont restés 4 ans et plus dans une ville sont regroupés dans `Stay_In_Current_City_Years`

In [9]:
df.Age.value_counts().sort_index()

Age
0-17      15102
18-25     99660
26-35    219587
36-45    110013
46-50     45701
51-55     38501
55+       21504
Name: count, dtype: int64

In [10]:
df.Stay_In_Current_City_Years.value_counts().sort_index()

Stay_In_Current_City_Years
0      74398
1     193821
2     101838
3      95285
4+     84726
Name: count, dtype: int64

In [11]:
# nombre de valeurs uniques par variables
df.nunique()

User_ID                        5891
Product_ID                     3631
Gender                            2
Age                               7
Occupation                       21
City_Category                     3
Stay_In_Current_City_Years        5
Marital_Status                    2
Product_Category_1               20
Product_Category_2               17
Product_Category_3               15
Purchase                      18105
dtype: int64

## 1.3 Gestion des valeurs manquantes

In [12]:
count_na = df.isnull().sum()

count_na[count_na != 0] / df.shape[0]

Product_Category_2    0.315666
Product_Category_3    0.696727
dtype: float64

Les valeurs manquantes se concentrent sur les catégories supplémentaires pour les produits qui doivent être optionnelles.
On a 32% de valeurs manquantes pour `Product_Category_2` et 70% pour `Product_Category_3`.

In [13]:
df['Product_Category_1'].value_counts().sort_index()

Product_Category_1
1     140378
2      23864
3      20213
4      11753
5     150933
6      20466
7       3721
8     113925
9        410
10      5125
11     24287
12      3947
13      5549
14      1523
15      6290
16      9828
17       578
18      3125
19      1603
20      2550
Name: count, dtype: int64

Comme les catégories sont encodées entre 1 et 20. On peut remplacer les valeurs manquantes par `0` ou `-1`.

In [14]:
df = df.fillna(0)

In [15]:
df[
    ['Product_Category_2', 'Product_Category_3']
] = df[['Product_Category_2', 'Product_Category_3']].astype(int)

## 1.4 Encodage des valeurs

Comme suggéré dans l'énoncé, nous allons encoder les variables catégorielles restantes.

In [26]:
# colonnes à encoder
cat_cols = [col for col in df.select_dtypes(exclude='number').columns if col != 'Product_ID']

In [27]:
# dictionnaire pour l'encoding
dict_encoding = {}
dict_decoding = {}

for col in cat_cols:
    values = sorted(df[col].unique())
    dict_encoding[col] = {v: i for i, v in enumerate(values)}
    dict_decoding[col] = {i: v for i, v in enumerate(values)}

dict_encoding

{'Gender': {'F': 0, 'M': 1},
 'Age': {'0-17': 0,
  '18-25': 1,
  '26-35': 2,
  '36-45': 3,
  '46-50': 4,
  '51-55': 5,
  '55+': 6},
 'City_Category': {'A': 0, 'B': 1, 'C': 2},
 'Stay_In_Current_City_Years': {'0': 0, '1': 1, '2': 2, '3': 3, '4+': 4}}

In [28]:
# encodage
df = df.replace(dict_encoding)
df[cat_cols] = df[cat_cols].astype(int)

In [ ]:



{'dict_encoding': dict_encoding, 'dict_decoding': dict_decoding}

In [97]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 550068 entries, 0 to 550067
Data columns (total 12 columns):
 #   Column                      Non-Null Count   Dtype
---  ------                      --------------   -----
 0   User_ID                     550068 non-null  int64
 1   Product_ID                  550068 non-null  str  
 2   Gender                      550068 non-null  int64
 3   Age                         550068 non-null  int64
 4   Occupation                  550068 non-null  int64
 5   City_Category               550068 non-null  int64
 6   Stay_In_Current_City_Years  550068 non-null  int64
 7   Marital_Status              550068 non-null  int64
 8   Product_Category_1          550068 non-null  int64
 9   Product_Category_2          550068 non-null  int64
 10  Product_Category_3          550068 non-null  int64
 11  Purchase                    550068 non-null  int64
dtypes: int64(11), str(1)
memory usage: 50.4 MB


In [98]:
df.describe().round(2)

,User_ID,Gender,Age,Occupation,City_Category,Stay_In_Current_City_Years,Marital_Status,Product_Category_1,Product_Category_2,Product_Category_3,Purchase
count,550068.00,550068.00,550068.00,550068.00,550068.00,550068.00,550068.00,550068.00,550068.00,550068.00,550068.00
mean,1003028.84,0.75,2.50,8.08,1.04,1.86,0.41,5.40,6.74,3.84,9263.97
std,1727.59,0.43,1.35,6.52,0.76,1.29,0.49,3.94,6.22,6.25,5023.07
min,1000001.00,0.00,0.00,0.00,0.00,0.00,0.00,1.00,0.00,0.00,12.00
25%,1001516.00,1.00,2.00,2.00,0.00,1.00,0.00,1.00,0.00,0.00,5823.00
50%,1003077.00,1.00,2.00,7.00,1.00,2.00,0.00,5.00,5.00,0.00,8047.00
75%,1004478.00,1.00,3.00,14.00,2.00,3.00,1.00,8.00,14.00,8.00,12054.00
max,1006040.00,1.00,6.00,20.00,2.00,4.00,1.00,20.00,18.00,18.00,23961.00


In [49]:
df['Gender'].value_counts()

Gender
M    414259
F    135809
Name: count, dtype: int64